# MLP TD-PINNs: Weak Form — Burgers Invíscida (ν=0), IC Gaussiana

**PDE:** $u_t + u\,u_x = 0$

**Esquema temporal:** Backward Euler con paso $\Delta t$ pequeño:
$$R^n = \frac{u^n - u^{n-1}}{\Delta t} + u^n\,\partial_x u^n$$

**Loss débil (Petrov–Galerkin):**
$$\mathcal{L}_{PDE} = \frac{1}{K}\sum_k\left(\frac{1}{N}\sum_i R^n(x_i)\,v_k(x_i)\right)^2$$

**Solución exacta:** Hopf–Lax (minimización discreta de $F(\xi)+(x-\xi)^2/(2t)$).

**Evaluación:** solo $t = 0.6$ (post-choque, $t_c \approx 0.412$).

In [ ]:
import sys, os, time
sys.path.insert(0, os.path.dirname(os.getcwd()))

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from scipy.special import erf as sp_erf

from pinns_framework import (
    PINArchitecture, PINSpatialDimension, PINLossFunction, PINTrainer,
    collocation_curvature, compute_error,
)

try:
    _trapz = np.trapezoid
except AttributeError:
    _trapz = np.trapz

torch.manual_seed(42)
np.random.seed(42)
print('torch:', torch.__version__)
DEVICE = 'cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu')
print('device:', DEVICE)

## 1. Solución exacta — Hopf–Lax

In [ ]:
X0, XF  = 0.0, 2 * np.pi
A_G     = 2.0
SIGMA_G = 0.5
X0_G    = np.pi
T_G     = 0.6
DT_G    = 0.05      # 12 pasos hasta T=0.6
NT_G    = round(T_G / DT_G)
T_EVAL  = T_G       # único tiempo de evaluación

t_c = SIGMA_G * np.sqrt(np.e) / A_G
print(f'Shock formation time  t_c = {t_c:.4f}')
print(f'Steps: {NT_G}  x  dt={DT_G}  →  T={T_G}')

_sqrt2   = np.sqrt(2.0)
_erf_ref = sp_erf(-X0_G / (SIGMA_G * _sqrt2))

def _F(xi, A=A_G, x0=X0_G, sigma=SIGMA_G):
    return A * sigma * np.sqrt(np.pi / 2) * (
        sp_erf((xi - x0) / (sigma * _sqrt2)) - _erf_ref
    )

def exact_solution(t, x_arr, A=A_G, x0=X0_G, sigma=SIGMA_G, n_xi=40000):
    """Entropy solution via Hopf-Lax."""
    x_arr = np.asarray(x_arr, dtype=float)
    if t == 0:
        return A * np.exp(-(x_arr - x0)**2 / (2 * sigma**2))
    xi_lo = min(X0, x0 - 10*sigma) - 1.0
    xi_hi = max(XF, x0 + A*t + 5*sigma) + 1.0
    xi    = np.linspace(xi_lo, xi_hi, n_xi)
    F_xi  = _F(xi)
    h     = F_xi[None, :] + (x_arr[:, None] - xi[None, :])**2 / (2 * t)
    return (x_arr - xi[h.argmin(axis=1)]) / t

x_plot = np.linspace(X0, XF, 600)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(x_plot, exact_solution(0,     x_plot), 'k-', lw=2, label='t=0 (IC)')
axes[0].plot(x_plot, exact_solution(T_EVAL, x_plot), 'r-', lw=2, label=f't={T_EVAL} (choque)')
axes[0].set_title('Exacta Hopf-Lax'); axes[0].legend(); axes[0].grid(alpha=0.3)
axes[0].set_xlabel('x'); axes[0].set_ylabel('u')

# Características
t_range = np.linspace(0, T_G, 200)
for xi0 in np.linspace(X0_G - 2.5*SIGMA_G, X0_G + 2.5*SIGMA_G, 30):
    u0 = A_G * np.exp(-(xi0 - X0_G)**2 / (2*SIGMA_G**2))
    axes[1].plot(xi0 + t_range*u0, t_range, lw=0.7, alpha=0.6, color='C0')
axes[1].axhline(t_c,    color='red',    lw=1.5, ls='--', label=f't_c={t_c:.3f}')
axes[1].axhline(T_EVAL, color='orange', lw=1.5, ls='-',  label=f't_eval={T_EVAL}')
axes[1].set_xlim(X0, XF); axes[1].set_ylim(0, T_G)
axes[1].set_xlabel('x'); axes[1].set_ylabel('t')
axes[1].set_title('Características'); axes[1].legend(); axes[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 2. IC / BC

In [ ]:
x_ic_np = np.linspace(X0, XF, 300)
u_ic_np = exact_solution(0, x_ic_np)

x_ic = torch.tensor(x_ic_np, dtype=torch.float32).unsqueeze(1)
u_ic = torch.tensor(u_ic_np, dtype=torch.float32).unsqueeze(1)

x_bc = torch.tensor([[X0], [XF]], dtype=torch.float32)
u_bc = torch.zeros(2, 1)

## 3. Arquitectura y Residual

Red: $x \to u$ (1 entrada, igual que el caso viscoso).

**Residual Backward Euler invíscido:**
$$R^n = \frac{u^n - u^{n-1}}{\Delta t} + u^n\,\partial_x u^n$$

Sin $\nu\,\partial_{xx}u$ — se ahorra la segunda derivada.

In [ ]:
N_COLLOC = 800
N_TEST   = 80

ARCH        = 'MLP'
PARAMS_ARCH = {'layers': (1, 64, 64, 64, 1), 'activation': 'tanh'}

def burgers_residual(u, u_prev, x, dt, nu):   # nu=0, ignorado
    du_dx = torch.autograd.grad(u.sum(), x, create_graph=True)[0]
    return (u - u_prev) / dt + u * du_dx

x_dim = PINSpatialDimension('x', domain=(X0, XF), n_colloc=N_COLLOC)
print(PINArchitecture(ARCH, PARAMS_ARCH))

## 4. Loss débil

In [ ]:
loss_weak = PINLossFunction([x_dim], burgers_residual, form='weak', n_test=N_TEST)
print(loss_weak)

## 5. Entrenamiento — TD-PINNs

In [ ]:
net = PINArchitecture(ARCH, PARAMS_ARCH)
trainer = PINTrainer(
    net=net, scheme='TD-PINNS', loss_fn=loss_weak,
    spatial_dims=[x_dim], epochs=20000, dt=DT_G,
    lr=1e-3, tolerance=0,
    ic_weight=15.0, bc_weight=5.0, pde_weight=1.0,
    collocation_fn=collocation_curvature,
    verbose=True, log_every=2000,
)

print('=== Inviscid Burgers (nu=0) --- TD-PINNs Weak Form ===')
t0   = time.perf_counter()
nets = trainer.fit(x_ic, u_ic, x_bc, u_bc, T=T_G, nu=0)
wall = time.perf_counter() - t0
print(f'Wall time: {wall:.1f} s   ({NT_G} pasos x {20000} epocas)')

## 6. Comparación vs exacta en t = 0.6

In [ ]:
dev = trainer.device
x_eval_np = np.linspace(X0, XF, 600)
x_eval    = torch.tensor(x_eval_np, dtype=torch.float32).unsqueeze(1).to(dev)

# nets[-1] es la red del último paso de tiempo (t = T_G = 0.6)
nets[-1].eval()
with torch.no_grad():
    u_pred = nets[-1](x_eval).cpu().numpy().ravel()

u_exact = exact_solution(T_EVAL, x_eval_np)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(x_eval_np, u_exact, 'k-',  lw=2,   label='Exacta (Hopf-Lax)')
ax.plot(x_eval_np, u_pred,  'r--', lw=1.5, label='MLP Debil (TD-PINNs)')
ax.set_title(f'Burgers inviscida (nu=0) --- t = {T_EVAL}  (post-choque,  t_c={t_c:.3f})', fontsize=11)
ax.set_xlabel('x'); ax.set_ylabel('u')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 7. Error en t = 0.6

In [ ]:
diff  = np.abs(u_pred - u_exact)
L1    = float(_trapz(diff,        x_eval_np))
L2    = float(np.sqrt(_trapz(diff**2, x_eval_np)))
Linf  = float(diff.max())
denom = float(np.sqrt(_trapz(u_exact**2, x_eval_np))) + 1e-12

pd.set_option('display.float_format', '{:.4e}'.format)
df_err = pd.DataFrame(
    [{'L1': L1, 'L2': L2, 'Linf': Linf, 'rel_L2': L2/denom}],
    index=[f't={T_EVAL}']
)
print(f'=== Error  t = {T_EVAL} ===')
print(df_err.to_string())

## 8. Evolución paso a paso

Muestra cómo la solución se va deformando en cada paso de tiempo hasta formar el choque.

In [ ]:
# Mostrar IC + cada 3 pasos + paso final
steps_to_show = [0] + list(range(3, NT_G+1, 3))
if NT_G not in steps_to_show:
    steps_to_show.append(NT_G)

ncols = len(steps_to_show)
fig, axes = plt.subplots(1, ncols, figsize=(3.5*ncols, 4), squeeze=False)
axes = axes[0]
cmap = plt.cm.plasma
for ax, step in zip(axes, steps_to_show):
    t_val = step * DT_G
    nets[step].eval()
    with torch.no_grad():
        u_nn = nets[step](x_eval).cpu().numpy().ravel()
    u_ex = exact_solution(t_val, x_eval_np)
    c = cmap(step / NT_G)
    ax.plot(x_eval_np, u_ex, 'k-',  lw=2,   label='Exacta')
    ax.plot(x_eval_np, u_nn, '--',  lw=1.5, color=c, label='MLP')
    ax.set_title(f't={t_val:.2f}', fontsize=9)
    ax.set_xlabel('x'); ax.set_ylabel('u'); ax.grid(alpha=0.25)
    if step == steps_to_show[0]:
        ax.legend(fontsize=7)
fig.suptitle(f'TD-PINNs Weak Form --- Burgers inviscida (nu=0)', fontsize=12)
plt.tight_layout(); plt.show()